# Reproducing Pakhomov et al. (2023) from dandiset 000875

**Paper.** Pakhomov AG, Gudvangen E, Mangalanathan U, Kondratiev O, Redondo L, Semenov I (2023).
Next generation CANCAN focusing for remote stimulation by nanosecond electric pulses.
*Bioelectrochemistry* 152, 108437.
[doi:10.1016/j.bioelechem.2023.108437](https://doi.org/10.1016/j.bioelechem.2023.108437). The
claims file was written from the accepted author manuscript in PubMed Central (PMC10247516), with
Methods, Results, Discussion and legends; there is no supplement and no linked code.

**Dandiset.** [DANDI:000875](https://dandiarchive.org/dandiset/000875/0.240515.1925), version
0.240515.1925. The paper cites no dandiset or version. This is the only published version, and
the draft lists the same nine assets (same asset ids and sizes), so the most recent published
version is pinned.

**What the dandiset holds.** Nine stitched YO-PRO-1 (YP) images of CHO monolayers, one per dish,
three dishes each after Packet 2 was rotated 1, 5 or 10 times over the four electrodes (1Px4,
5Px4, 10Px4; 4, 20 or 40 packets), first pulse 3.2 kV. This is the experiment of Fig. 6. The
single-cell time-lapse of Fig. 4 and the monolayer experiments of Figs. 5, 7, 8 and 9 are not in
it, and the Sim4Life model of Fig. 3 is not public.

**Method.** Each image is streamed from the archive. The four electrode footprints are found in
an 8 x 8 block-mean overview, and mean intensities are measured in square ROIs along the
diagonals of the full-resolution image, over +/- 1.65 mm from the array center (Methods 2.4, small
array), under every reading the claims file names. The background reading b2 subtracts a dark offset
estimated from the darkest parts of each image, since the electrode footprints turned out to hold
cells. The pixel size is not in the files; 0.65 um
per pixel (camera pixel through the 10x objective) is assumed, and checked against the electrode
geometry. Helper: `cancan.py`.

| Claim | Status |
|---|---|
| fig6a-more-packets-more-uptake | tested |
| fig6b-peak-relative-to-periphery | tested |
| fig6-trough-half-of-center | tested |
| fig6-peak-narrower | tested |
| fig4-bipolar-cancellation, fig4-twenty-fold-cancellation | blocked: data missing (single-cell time-lapse not in the dandiset) |
| fig5a-packet-without-rotation, fig5bc-rotation-focuses-center | blocked: data missing (Fig. 5 dishes not in the dandiset) |
| fig7ab-amplitude-scales-uptake, fig7-targeting-independent-of-amplitude, fig7-targeting-independent-of-array-size | blocked: data missing (only 3.2 kV, small array, 1Px4 to 10Px4) |
| fig8-packet2-best-targeting, fig8b-normalized-center, fig8-cancellation-predicts-targeting | blocked: data missing (Packets 1 and 3, 4 kV, not in the dandiset; the last also needs Fig. 4) |
| fig9-gap-fades-periphery, fig9b-six-fold-targeting | blocked: data missing (no lifted-array dishes) |
| fig3-field-simulation | blocked: data missing (Sim4Life model not public) |

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import cancan

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 400)
warnings.filterwarnings("ignore", category=RuntimeWarning)
TREATMENTS = ["1Px4", "5Px4", "10Px4"]
HIST_BINS = np.geomspace(20, 5000, 120)  # log-spaced bins for the intensity histograms, a plotting choice

started = time.time()
assets = cancan.list_assets()
dishes = {t: [] for t in TREATMENTS}
for a in assets:
    meta = cancan.read_metadata(a["asset_id"])
    overview = cancan.load_overview(a["asset_id"])
    electrodes = cancan.find_electrodes(overview)
    geo = cancan.geometry(electrodes)
    region, origin = cancan.load_region(a["asset_id"], geo["center"], 2700)
    profiles = cancan.dish_profiles(region, origin, electrodes)
    half_geo = cancan.geometric_half_span(electrodes)
    profiles_geo = cancan.dish_profiles(region, origin, electrodes, half_geo)
    k = 32  # an 80 x 80 thumbnail (blocks of 256 x 256 pixels), kept for the duplicate check and the figure
    h = overview.shape[0] // k * k
    thumb = overview[:h, :h].reshape(h // k, k, h // k, k).mean(axis=(1, 3))
    zones = cancan.zone_values(overview, electrodes)
    dark = cancan.dark_level(overview)
    R = np.mean([np.hypot(e["row"] - geo["center"][0], e["col"] - geo["center"][1]) for e in electrodes])
    dy, dx = np.nonzero((overview > 0) & (overview <= dark))
    dark_far = float((np.hypot(dy * cancan.BLOCK - geo["center"][0], dx * cancan.BLOCK - geo["center"][1]) > 2 * R).mean())
    # The array area of the overview (8 x 8 blocks) and a full-resolution crop of footprint 0 and its rim, for the figures.
    cb = (geo["center"] / cancan.BLOCK).astype(int)
    hw = int(1.45 * np.max([np.hypot(e["row"] - geo["center"][0], e["col"] - geo["center"][1]) for e in electrodes]) / cancan.BLOCK)
    array_crop = overview[max(0, cb[0] - hw):cb[0] + hw, max(0, cb[1] - hw):cb[1] + hw].copy()
    e0 = electrodes[0]
    fp_crop, fp_origin = cancan.load_region(a["asset_id"], (e0["row"], e0["col"]), int(1.6 * e0["radius"]))
    # Full-resolution 600 x 600 patches: the footprint center, and the monolayer 1.25 radii out, away from the array.
    u = np.array([e0["row"], e0["col"]]) - geo["center"]
    u /= np.linalg.norm(u)
    patches = {}
    for name, p in {"footprint": (e0["row"], e0["col"]), "outside": np.array([e0["row"], e0["col"]]) + 1.25 * e0["radius"] * u}.items():
        py, px = int(p[0]) - fp_origin[0], int(p[1]) - fp_origin[1]
        patches[name] = fp_crop[py - 300:py + 300, px - 300:px + 300].copy()
    s4 = fp_crop.shape[0] // 4 * 4, fp_crop.shape[1] // 4 * 4
    fp_small = fp_crop[:s4[0], :s4[1]].reshape(s4[0] // 4, 4, s4[1] // 4, 4).mean(axis=(1, 3))
    dishes[a["treatment"]].append({
        **a, "meta": meta, "electrodes": electrodes, "geometry": geo, "profiles": profiles,
        "profiles_geo": profiles_geo, "half_span_geo_mm": half_geo * cancan.PIXEL_UM / 1000,
        "footprint": cancan.footprint_level(overview, electrodes),
        "dark": dark, "dark_far_fraction": dark_far, "region_min": float(region.min()),
        "dark_q": {q: cancan.dark_level(overview, q) for q in (0.1, 1.0, 5.0)},
        "zone_hist": {z: np.histogram(v, HIST_BINS)[0] for z, v in zones.items()},
        "zone_n": {z: len(v) for z, v in zones.items()},
        "zone_pct": {z: np.percentile(v, [1, 5, 50, 95, 99]) for z, v in zones.items()},
        "overview_stats": {"min": float(overview.min()), "max": float(overview.max()),
                           "median_nonzero": float(np.median(overview[overview > 0])),
                           "zero_fraction": float((overview == 0).mean())},
        "region_stats": {"min": int(region.min()), "max": int(region.max()), "median": float(np.median(region)),
                         "near_ceiling_fraction": float((region >= 65000).mean())},
        "thumb": thumb, "array_crop": array_crop, "array_crop_origin": (max(0, cb[0] - hw), max(0, cb[1] - hw)),
        "fp_small": fp_small, "fp_crop_origin": fp_origin, "patches": patches,
    })
    del overview, region, fp_crop
print(f"loaded {len(assets)} files from dandiset {cancan.DANDISET_ID} version {cancan.DANDISET_VERSION} "
      f"in {time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
rows = []
for t in TREATMENTS:
    for d in dishes[t]:
        m = d["meta"]
        desc = m["session_description"]
        cycles = int(desc.split("A total of ")[1].split(" cycles")[0])
        kv = float(desc.split("single ")[1].split(" kV")[0])
        rows.append({"treatment": t, "dish": d["dish"], "subject_id": m["subject_id"],
                     "cycles_in_description": cycles, "kV_in_description": kv, "kV_in_name": d["kv_from_name"],
                     "session_start": m["session_start_time"], "species": m["species"].split("/")[-1],
                     "strain": m["strain"], "age": m["age"], "shape": m["shape"], "unit": m["unit"],
                     "rate_Hz": m["rate"], "series": m["series_fields"], "size_MB": round(d["size"] / 1e6)})
info = pd.DataFrame(rows)
print("Files per treatment:", info.treatment.value_counts().reindex(TREATMENTS).to_dict())
print(info.drop(columns=["series"]).to_string(index=False))
print("\nPacket cycles in the description equal the packets per electrode in the name:",
      bool((info.cycles_in_description == info.treatment.str[:-3].astype(int)).all()))
print("First-pulse voltage in every description:", sorted(info.kV_in_description.unique()), "kV; in the names:",
      sorted(info.kV_in_name.unique()))
print("Fields of the ImageSeries:", info.series.iloc[0], "; groups in the file:", dishes["1Px4"][0]["meta"]["groups"])
shared = info.groupby("session_start").dish.apply(list)
print("\nSession start times shared by more than one dish:", shared[shared.map(len) > 1].to_dict())
print("Device description:", dishes["1Px4"][0]["meta"]["device_description"])
print("Experimenter:", dishes["1Px4"][0]["meta"]["experimenter"])
vm = cancan.version_metadata()
summ = vm["assetsSummary"]
print("\nArchive summary of the version: numberOfFiles", summ["numberOfFiles"], "; numberOfSubjects", summ["numberOfSubjects"],
      "; species", [x["name"] for x in summ.get("species", [])], "; distinct subject ids in the files", info.subject_id.nunique())
print("Dandiset description:", vm["description"])

The nine files are three dishes each of 1Px4, 5Px4 and 10Px4 (Packet 2 applied 1, 5 or 10 times
from each electrode, 4, 20 or 40 packets in all), first pulse 3.2 kV, as the session
descriptions say and the file names agree. This is the experiment of Fig. 6 (Results 3.3,
paragraph 1); the legend gives n = 3 to 4, and the dandiset has 3 per treatment. Each file holds
one stitched image and nothing else: no ROI table, no pixel size, no electrode positions. The
species is NCBITaxon 9913 (*Bos taurus*, cattle), while the strain field and the paper say CHO
cells (Chinese hamster); the age "P1D" has no meaning for a cell line. The ImageSeries has
unit "n.a." and a rate of 2 Hz for a single frame. Two pairs of dishes share a session start
time to the second, one pair across treatments (dish 09 of 5Px4 and dish 25 of 1Px4); the images
themselves differ (checked below), so the start times are not acquisition times of each dish.
The "200ms" in every file name is not explained; packets were given at 1 Hz.

In [ ]:
rows = []
for t in TREATMENTS:
    for d in dishes[t]:
        rows.append({"treatment": t, "dish": d["dish"], **{f"overview_{k}": round(v, 3) for k, v in d["overview_stats"].items()},
                     **{f"region_{k}": round(v, 5) if isinstance(v, float) else v for k, v in d["region_stats"].items()},
                     "footprint_median": round(d["footprint"], 1),
                     **{f"dark_p{q:g}": round(v, 1) for q, v in d["dark_q"].items()},
                     "roi_saturated_max": max(float(p["saturated"].max()) for p in d["profiles"].values())})
vals = pd.DataFrame(rows)
print("Intensities are uint16 camera counts (unit 'n.a.', conversion 1, offset 0).")
print("overview_*: 8 x 8 block means of the whole 20480 x 20480 image; region_*: the full-resolution square around the array center;")
print("dark_p*: percentiles of the nonzero block means of the whole image.")
print(vals.to_string(index=False))

Pixels are 16-bit counts; the zero pixels (26% of each image) are the padding outside the
stitched tiles. The brightest pixels reach about 65440, the ceiling of these images; the fraction
of pixels at or above 65000 in the central region is printed as `region_near_ceiling_fraction`,
and the largest such fraction in any ROI as `roi_saturated_max`: none in 1Px4, up to 5.0% of
an ROI in 10Px4, so the brightest 10Px4 ROIs, near the electrodes, are slightly capped. This
lowers the 10Px4 periphery, which works against the order tested in fig6a. The median inside the
electrode footprints is 239 to 403 counts; the darkest pixel of the central region is 110 to 200
counts; the 1st percentile of each image's nonzero block means is 73.5 to 87.1 counts (0.1st
percentile 58.2 to 60.7, 5th 99.2 to 153.1). These are the candidate background levels examined
below.

In [ ]:
rows = []
for t in TREATMENTS:
    for d in dishes[t]:
        g = d["geometry"]
        rows.append({"treatment": t, "dish": d["dish"], "center_row": round(g["center"][0]), "center_col": round(g["center"][1]),
                     "footprint_diam_mm": np.round(g["footprint_diameter_mm"], 2).tolist(),
                     "diag_center_to_center_mm": np.round(g["diag_cc_mm"], 2).tolist(),
                     "diag_between_footprints_mm": np.round(g["diag_inner_mm"], 2).tolist(),
                     "sides_mm": np.round(g["side_cc_mm"], 2).tolist()})
geo = pd.DataFrame(rows)
print(f"Distances at the assumed {cancan.PIXEL_UM} um per pixel:")
print(geo.to_string(index=False))
inner = np.concatenate([d["geometry"]["diag_inner_mm"] for t in TREATMENTS for d in dishes[t]])
diam = np.concatenate([d["geometry"]["footprint_diameter_mm"] for t in TREATMENTS for d in dishes[t]])
cc = np.concatenate([d["geometry"]["diag_cc_mm"] for t in TREATMENTS for d in dishes[t]])
print(f"\nFootprint diameter {diam.min():.2f} to {diam.max():.2f} mm (median {np.median(diam):.2f}); needle diameter 1.5 mm (Methods 2.3).")
print(f"Diagonal between footprint edges {inner.min():.2f} to {inner.max():.2f} mm.")
print(f"Small array: ROIs cover +/- 1.65 mm = 3.30 mm; the margin left at each end is {(inner.min() - 3.3) / 2:.2f} to {(inner.max() - 3.3) / 2:.2f} mm,"
      f" {100 * (inner.min() - 3.3) / 2 / inner.min():.1f}% to {100 * (inner.max() - 3.3) / 2 / inner.max():.1f}% of the diagonal (Methods 2.4: '<5% of the diagonal length' excluded).")
print(f"Large array: ROIs cover +/- 3.65 mm = 7.30 mm, more than the {inner.max():.2f} mm between footprints at this scale.")
hg = [d["half_span_geo_mm"] for t in TREATMENTS for d in dishes[t]]
print(f"Variant extent, 5% of the footprint-to-footprint diagonal short of each footprint: +/- {min(hg):.2f} to {max(hg):.2f} mm at this scale (main: +/- 1.65 mm).")

# Scale check against the array sizes of the paper (Methods 2.3, 2.4; Section 3.5).
print("\nScale check. The paper: large array 8.4 mm diagonal (Section 3.5); 'approximately a 2-fold size difference' (Methods 2.3);")
print("needles 1.5 mm (Methods 2.3); ROIs over +/- 1.65 mm (small) and +/- 3.65 mm (large), '<5% of the diagonal length' excluded (Methods 2.4).")
print(f"  If 8.4 mm were center to center, the large array would leave {8.4 - 1.5:.2f} mm between needle edges, less than its own 7.30 mm of ROIs;"
      " so the paper's ROI extent fits only if the diagonal is read between needle edges.")
print(f"  Read between needle edges, the large array's ROIs leave {100 * (8.4 - 7.3) / 2 / 8.4:.1f}% of the diagonal at each end; the same"
      f" proportion gives a small-array diagonal of {3.3 / (7.3 / 8.4):.2f} mm, and half of 8.4 mm is 4.20 mm.")
for scale in [cancan.PIXEL_UM, 0.53]:
    f = scale / cancan.PIXEL_UM
    edge = cc * f - 1.5
    print(f"  At {scale} um per pixel: center-to-center diagonal {cc.min() * f:.2f} to {cc.max() * f:.2f} mm; between 1.5-mm needle edges"
          f" {edge.min():.2f} to {edge.max():.2f} mm; between dark footprint edges {inner.min() * f:.2f} to {inner.max() * f:.2f} mm;"
          f" the 3.30 mm of small-array ROIs fit between the footprints: {bool(inner.min() * f > 3.3)}.")
scale_needle = 1.5 / np.median(diam) * cancan.PIXEL_UM
print(f"  Scale that makes the median footprint exactly 1.5 mm: {scale_needle:.3f} um per pixel; scale that makes the center-to-center diagonal"
      f" 4.2 mm (half of 8.4 mm read center to center): {4.2 / np.median(cc) * cancan.PIXEL_UM:.3f} um per pixel.")

The files give no pixel size. At 0.65 um per pixel (the 6.5 um pixel of the ORCA-Flash4 camera
through the 10x objective, both named in the file's device description and Methods 2.4), the four
footprints sit on a near square with 5.23 to 5.55 mm center-to-center diagonals and 3.40 to 3.71
mm between the dark footprint edges. The small array's +/- 1.65 mm of ROIs then fits between the
footprints with 1.5 to 5.6% of the diagonal left at each end, close to the 5% the Methods
exclude, and the large array's +/- 3.65 mm does not fit at all, so the images are of the small
array.

Two cues point to a smaller scale. The dark footprints are 1.66 to 2.09 mm across, against
1.5-mm needles (0.532 um per pixel would make the median footprint 1.5 mm), and half of the
large array's 8.4 mm diagonal, read center to center, is 4.2 mm (0.510 um per pixel). The scale
check above weighs them against the paper's own ROI extent. Read center to center, 8.4 mm would
leave 6.90 mm between the large array's needles, less than the 7.30 mm its ROIs cover, so the
paper's diagonal is between needle edges; on that reading the large array's ROIs leave 6.5% of
it at each end, and the same proportion gives a small-array diagonal of 3.80 mm (half of 8.4 mm
is 4.20 mm). At 0.65 um per pixel the small array measures 3.73 to 4.05 mm between 1.5-mm needle
edges, which agrees; at 0.53 um it measures 2.77 to 3.03 mm, too short for the paper's 3.30 mm of
ROIs. So the camera scale is kept, as an assumption, and the dark footprints are taken to be wider
than the needles. The variant that sets the ROI extent from the footprints uses no pixel size,
and each claim gives its outcome under it.

In [ ]:
names = [(t, d["dish"]) for t in TREATMENTS for d in dishes[t]]
thumbs = [np.log(np.clip(d["thumb"], 1, None)) for t in TREATMENTS for d in dishes[t]]
corr = np.array([[np.corrcoef(a.ravel(), b.ravel())[0, 1] for b in thumbs] for a in thumbs])
identical = [(names[i], names[j]) for i in range(len(thumbs)) for j in range(i + 1, len(thumbs))
             if np.array_equal(thumbs[i], thumbs[j])]
print("Pairs of identical images:", identical if identical else "none")
off = corr[~np.eye(len(corr), dtype=bool)]
print(f"Correlation of log thumbnails between different dishes: {off.min():.3f} to {off.max():.3f}")
centers = np.array([d["geometry"]["center"] for t in TREATMENTS for d in dishes[t]])
print("Array centers (row, col) differ between dishes:", len({tuple(np.round(c, -1)) for c in centers}) == len(centers))
pair_corr = {f"{names[i][0]} dish{names[i][1]} / {names[j][0]} dish{names[j][1]}": round(corr[i, j], 3)
             for i in range(len(names)) for j in range(i + 1, len(names))
             if {names[i][1], names[j][1]} in ({20, 22}, {9, 25})}
print("Dishes that share a session start time, correlation of thumbnails:", pair_corr)

fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for ax, (t, d) in zip(axes.flat, [(t, d) for t in TREATMENTS for d in dishes[t]]):
    th = d["thumb"]
    ax.imshow(np.clip(th, 0, np.percentile(th[th > 0], 99)), cmap="gray")
    c = np.array([[e["row"], e["col"]] for e in d["electrodes"]]) / 256
    for e in d["electrodes"]:
        ax.add_patch(plt.Circle((e["col"] / 256, e["row"] / 256), e["radius"] / 256, fill=False, color="y"))
    cen = c.mean(0)
    half = cancan.HALF_SPAN_MM * 1000 / cancan.PIXEL_UM / 256
    for a_, b_ in [(0, 2), (1, 3)]:
        u = (c[b_] - c[a_]) / np.linalg.norm(c[b_] - c[a_])
        ax.plot([cen[1] - half * u[1], cen[1] + half * u[1]], [cen[0] - half * u[0], cen[0] + half * u[0]], "c-", lw=1)
    ax.set_title(f"{t} dish {d['dish']}", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("Thumbnails (blocks of 256 x 256 pixels), detected footprints (yellow) and the ROI lines (cyan)")
plt.tight_layout(); plt.show()
print("Display: each thumbnail clipped at its 99th percentile of nonzero blocks.")

No two images are identical, and the array sits at a different place in each, so no recording was
deposited twice; the shared start times are a metadata problem only. The footprints and ROI lines
drawn on the thumbnails show the detection. Which electrode is e1 cannot be told from the images,
but every treatment here rotates the packet over all four electrodes, so the diagonals are
equivalent; diagonal A joins footprints 0 and 2 in the order of their angle around the center.

There is no time axis or trial table to align: each file is one image taken after the
treatment. The check that the treatment labels belong to the images is the expected contrast
between treatments, which is the subject of the first claim below.

### What the electrode footprints hold, and the dark offset

Reading b2 of the claims file subtracts "a cell-free or dark offset, where the data hold one".
The first round used the median inside the dark electrode footprints. The cells below show what
the footprints hold at full resolution, the intensity distributions of the footprints, of the
monolayer outside the array and of the exposed area between the electrodes, and where the darkest
blocks of each image lie.

In [ ]:
from scipy.ndimage import gaussian_filter

alld = [(t, d) for t in TREATMENTS for d in dishes[t]]
fig, axes = plt.subplots(3, 3, figsize=(13, 13))
for ax, (t, d) in zip(axes.flat, alld):
    im = d["array_crop"]
    o = d["array_crop_origin"]
    ax.imshow(np.log(np.clip(im, 1, None)), cmap="gray", vmin=np.log(np.percentile(im[im > 0], 1)), vmax=np.log(np.percentile(im[im > 0], 99.5)))
    c = np.array([[e["row"], e["col"]] for e in d["electrodes"]]) / cancan.BLOCK - np.array(o)
    for e in d["electrodes"]:
        ax.add_patch(plt.Circle((e["col"] / cancan.BLOCK - o[1], e["row"] / cancan.BLOCK - o[0]), e["radius"] / cancan.BLOCK, fill=False, color="y", lw=0.8))
    cen = c.mean(0)
    half = cancan.HALF_SPAN_MM * 1000 / cancan.PIXEL_UM / cancan.BLOCK
    for a_, b_ in [(0, 2), (1, 3)]:
        u = (c[b_] - c[a_]) / np.linalg.norm(c[b_] - c[a_])
        ax.plot([cen[1] - half * u[1], cen[1] + half * u[1]], [cen[0] - half * u[0], cen[0] + half * u[0]], "c-", lw=0.8)
    ax.set_title(f"{t} dish {d['dish']}", fontsize=9); ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("Array area at 8 x 8 block resolution (5.2 um per block at the assumed scale), log scale, clipped at the 1st and 99.5th percentiles:\n"
             "fitted footprint circles (yellow) and the ROI lines over +/- 1.65 mm (cyan)")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(3, 3, figsize=(13, 13))
for ax, (t, d) in zip(axes.flat, alld):
    im = d["fp_small"]
    o = d["fp_crop_origin"]
    e = d["electrodes"][0]
    ax.imshow(im, cmap="gray", vmin=np.percentile(im, 1), vmax=np.percentile(im, 99.5))
    ax.add_patch(plt.Circle(((e["col"] - o[1]) / 4, (e["row"] - o[0]) / 4), e["radius"] / 4, fill=False, color="y", lw=0.8))
    ax.set_title(f"{t} dish {d['dish']}: footprint 0", fontsize=9); ax.set_xticks([]); ax.set_yticks([])
plt.suptitle("Footprint 0 of each dish and its rim (full resolution shown in 4 x 4 pixel means), linear scale clipped at the 1st and 99.5th percentiles")
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(3, 6, figsize=(18, 9.5))
rows = []
for k, (t, d) in enumerate(alld):
    for j, name in enumerate(["footprint", "outside"]):
        z = d["patches"][name]
        ax = axes.flat[2 * k + j]
        ax.imshow(z, cmap="gray", vmin=np.percentile(z, 1), vmax=np.percentile(z, 95))
        ax.set_title(f"{t} d{d['dish']}: {name}", fontsize=8); ax.set_xticks([]); ax.set_yticks([])
    # Band-pass (difference of Gaussians, sigma 3 and 15 pixels, about 2 and 10 um): keeps structure of the size of nuclei, removes shot noise and gradients.
    tex = {name: gaussian_filter(d["patches"][name].astype(float), 3) - gaussian_filter(d["patches"][name].astype(float), 15) for name in ["footprint", "outside"]}
    rows.append({"treatment": t, "dish": d["dish"],
                 **{f"{name}_median": round(float(np.median(d["patches"][name])), 1) for name in tex},
                 **{f"{name}_texture_counts": round(float(np.subtract(*np.percentile(tex[name], [75, 25]))), 1) for name in tex}})
plt.suptitle("Full-resolution 600 x 600 pixel patches (0.39 mm at the assumed scale): the center of footprint 0, and the monolayer 1.25 radii\n"
             "from its center on the side away from the array; each clipped at its own 1st and 95th percentiles (the brightest debris saturates)")
plt.tight_layout(); plt.show()
print("Texture: interquartile range (counts) of the band-passed patch (difference of Gaussians, sigma 3 and 15 pixels), structure the size of nuclei.")
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(14, 10), sharex=True)
centers = np.sqrt(HIST_BINS[:-1] * HIST_BINS[1:])
rows = []
for ax, (t, d) in zip(axes.flat, alld):
    for z, col in [("footprint", "k"), ("outside", "tab:green"), ("between", "tab:red")]:
        h = d["zone_hist"][z]
        ax.plot(centers, h / h.sum(), color=col, label=z)
    for v, ls, lab in [(d["dark"], "--", "b2-dark"), (d["region_min"], ":", "b2-min")]:
        ax.axvline(v, color="gray", ls=ls, label=lab)
    ax.set_xscale("log"); ax.set_title(f"{t} dish {d['dish']}", fontsize=9)
    rows.append({"treatment": t, "dish": d["dish"],
                 **{f"{z}_p1/p50/p99": " / ".join(f"{x:.0f}" for x in d["zone_pct"][z][[0, 2, 4]]) for z in ["footprint", "outside", "between"]},
                 "dark_p1": round(d["dark"], 1), "region_min": d["region_min"],
                 "darkest_1pct_beyond_2R": round(d["dark_far_fraction"], 3)})
axes.flat[0].legend(fontsize=8)
for ax in axes[-1]:
    ax.set_xlabel("block mean (counts)")
plt.suptitle("Distributions of 8 x 8 block means: inside the footprints (within 0.7 radius), the monolayer outside the array\n"
             "(1.2 to 1.6 times the center-to-footprint distance), and the exposed area between the electrodes (within half that distance)")
plt.tight_layout(); plt.show()
print(pd.DataFrame(rows).to_string(index=False))
for z in ["footprint", "outside", "between"]:
    frac = [d["zone_hist"][z].sum() / d["zone_n"][z] for _, d in alld]
    print(f"{z}: fraction of blocks within the plotted range {HIST_BINS[0]:.0f} to {HIST_BINS[-1]:.0f} counts, {min(frac):.3f} to {max(frac):.3f}")
fo = np.array([d["footprint"] / d["zone_pct"]["outside"][2] for _, d in alld])
print(f"\nFootprint median over the median of the monolayer outside the array: {fo.min():.2f} to {fo.max():.2f}")
print("Darkest 1% of nonzero blocks farther than 2 center-to-footprint distances from the array center (off the glass bottom):",
      f"{min(d['dark_far_fraction'] for _, d in alld):.3f} to {max(d['dark_far_fraction'] for _, d in alld):.3f}")
print(f"b2-dark offsets {min(d['dark'] for _, d in alld):.1f} to {max(d['dark'] for _, d in alld):.1f} counts; "
      f"b2-min offsets {min(d['region_min'] for _, d in alld):.0f} to {max(d['region_min'] for _, d in alld):.0f}; "
      f"footprint medians {min(d['footprint'] for _, d in alld):.0f} to {max(d['footprint'] for _, d in alld):.0f}.")
fd = np.array([d["footprint"] / d["dark"] for _, d in alld])
print(f"Footprint median over the b2-dark offset: {fd.min():.1f} to {fd.max():.1f}")

The footprints are not cell-free. At full resolution several show a faint monolayer, with
structure the size of nuclei (band-passed interquartile range 7.6 to 21.4 counts, against 28.8 to
201.9 in the monolayer just outside), and their median is 0.74 to 0.94 of the median of the
monolayer outside the array and 3.1 to 5.0 times the darkest blocks. They read as cells that took
up little dye, so the footprint median is not the "cell-free or dark offset" of reading b2. It is
kept only as the variant `b2-fp`, which counts towards no outcome.

The darkest 1% of each image's nonzero blocks lie, for 99.6 to 100% of them, more than twice the
center-to-footprint distance from the array center, outside the circular edge visible in the
thumbnails, where no monolayer is in focus. Their upper bound, the 1st percentile (73.5 to 87.1
counts), is taken as the dark offset `b2-dark`; the darkest pixel of the region around the array
(110 to 200 counts) is a second reading, `b2-min`. The files hold no dark frame, so both are
estimates, and fig6b shows how its result depends on the size of the offset.

In [ ]:
def table(layout, background, extent="camera"):
    """Per-dish ROI values for a layout and background reading, by treatment.

    extent "camera": ROIs over +/- 1.65 mm at the assumed pixel size (main); "geometric": the
    variant that ends the ROIs 5% of the diagonal short of the footprints.
    """
    key = "profiles" if extent == "camera" else "profiles_geo"
    return {t: cancan.dish_matrix([{"profiles": d[key], "footprint": d["footprint"], "dark": d["dark"], "region_min": d["region_min"]}
                                   for d in dishes[t]], layout, background)
            for t in TREATMENTS}

pos, _ = table("r2", "b1")["1Px4"]
for lay in cancan.LAYOUTS:
    p, _ = table(lay, "b1")["1Px4"]
    print(f"{lay}: {len(p)} ROIs, {p[0]:.3f} to {p[-1]:.3f} mm, spacing {1000 * (p[1] - p[0]):.1f} um")
rows = []
for t in TREATMENTS:
    for lay in cancan.LAYOUTS:
        _, P = table(lay, "b1")[t]
        rows.append({"treatment": t, "layout": lay, "roi_min": P.min().round(1), "roi_max": P.max().round(1),
                     "dish_means": np.round(P.mean(axis=1), 1).tolist()})
print(pd.DataFrame(rows).to_string(index=False))

### The readings of the claims file

The claims file leaves the ROI layout, background, center, periphery and normalization order
open and names readings for each (header of the claims file); a claim holds when it holds under
any combination, and every combination is shown. The readings, as implemented in `cancan.py`:

- layout: `r2` 46 ROIs on each diagonal, averaged; `r1-dA` and `r1-dB` 92 ROIs on one diagonal;
- background: `b1` raw counts; `b2` "minus a cell-free or dark offset, where the data hold one",
  run two ways because the files hold no dark frame: `b2-dark` minus the 1st percentile of the
  image's nonzero 8 x 8 block means (blocks off the glass bottom, with no monolayer in focus), and
  `b2-min` minus the darkest pixel of the full-resolution region around the array;
- center: `c1` the two ROIs nearest the center (equally near, averaged); `c2` the highest ROI
  between the minima on either side;
- periphery: `e1` the mean of the two end ROIs; `e2-neg` and `e2-pos` one end;
- normalization: `n1` each dish normalized, then averaged; `n2` the mean profile normalized;
- least affected area: `t1` the mean of the two side minima; `t2-neg`, `t2-pos` one side.

The first round used the median inside the electrode footprints as b2. The footprints hold a
faint monolayer (shown above), so that level is not cell-free; it is shown as the variant
`b2-fp`, and it does not count towards any outcome.

The side minima and the `c2` peak are found on the treatment's mean profile. The reading shown
first is `r2`, `b1`, `c1`, `e1`, `n1`, `t1`: both diagonals (Fig. 5B legend averages both, and
these treatments rotate the packet), no background (none is named in Methods 2.4) and the most
literal center and periphery.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
colors = {"1Px4": "k", "5Px4": "tab:blue", "10Px4": "tab:red"}
data = table("r2", "b1")
ranges = []
for t in TREATMENTS:
    pos, P = data[t]
    m, se = P.mean(0), P.std(0, ddof=1) / np.sqrt(len(P))
    axes[0].errorbar(pos, m, se, color=colors[t], label=f"{t} (n={len(P)})", lw=1, capsize=0)
    N = P / cancan.periphery(P, "e1")[:, None] * 100
    mn, sn = N.mean(0), N.std(0, ddof=1) / np.sqrt(len(N))
    axes[1].errorbar(pos, mn, sn, color=colors[t], label=t, lw=1, capsize=0)
    ranges.append({"treatment": t, "au_mean_min": m.min().round(1), "au_mean_max": m.max().round(1),
                   "norm_mean_min": mn.min().round(1), "norm_mean_max": mn.max().round(1)})
axes[0].set_ylabel("YP intensity (counts)"); axes[1].set_ylabel("% of most peripheral ROIs (e1, n1)")
for ax in axes:
    ax.set_xlabel("distance from array center along the diagonal (mm)"); ax.legend()
axes[0].set_title("Reading r2, b1: mean +/- s.e. over dishes")
axes[1].set_title("Normalized as in Fig. 6B")
plt.tight_layout(); plt.show()
print(pd.DataFrame(ranges).to_string(index=False))
print("All points are within the axes.")

In [ ]:
# Per-dish profiles, reading r2, b1: raw counts (top) and in percent of the mean of the two ends (bottom).
fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True)
data = table("r2", "b1")
rows = []
for j, t in enumerate(TREATMENTS):
    pos, P = data[t]
    for d, p in zip(dishes[t], P):
        n = p / cancan.periphery(p, "e1")[0] * 100
        axes[0, j].plot(pos, p, lw=1, label=f"dish {d['dish']}")
        axes[1, j].plot(pos, n, lw=1, label=f"dish {d['dish']}")
        ci = cancan.center_index(p, "c1")
        rows.append({"treatment": t, "dish": d["dish"], "counts_min": round(p.min()), "counts_max": round(p.max()),
                     "neg_end": round(p[0]), "center_c1": round(cancan.center_value(p, ci)[0]), "pos_end": round(p[-1]),
                     "pct_min": round(n.min(), 1), "pct_max": round(n.max(), 1)})
    axes[0, j].plot(pos, P.mean(0), "k--", lw=1.5, label="mean")
    axes[1, j].plot(pos, (P / cancan.periphery(P, "e1")[:, None] * 100).mean(0), "k--", lw=1.5, label="mean (n1)")
    axes[0, j].set_title(t); axes[0, j].legend(fontsize=8)
axes[0, 0].set_ylabel("YP intensity (counts)"); axes[1, 0].set_ylabel("% of the mean of the two ends")
for ax in axes[1]:
    ax.set_xlabel("distance from array center (mm)")
plt.tight_layout(); plt.show()
print(pd.DataFrame(rows).to_string(index=False))
print("All points are within the axes.")

## fig6a-more-packets-more-uptake

> Increasing the total number of rotated Packet 2 packets from 4 (1Px4) to 20 (5Px4) and 40
> (10Px4) profoundly enhanced YP uptake throughout the electrode array.

Criterion (claims file): in a.u., at the periphery, the center and the minimum between them,
the mean is ordered 4 < 20 < 40 packets and 40 exceeds 4 by a two-sided Student's t-test on
per-dish values at p < 0.05; partially reproduced when the order holds at all three places and
the test at one or two. Run under every combination of layout, background, center, periphery
and trough readings (the normalization readings do not apply to a.u. values).

In [ ]:
rows = []
for lay in cancan.LAYOUTS:
    for bg in cancan.BACKGROUNDS + cancan.VARIANT_BACKGROUNDS:
        data = table(lay, bg)
        for cen in cancan.CENTERS:
            for per in cancan.PERIPHERIES:
                for tro in cancan.TROUGHS:
                    v = {}
                    for t in TREATMENTS:
                        pos, P = data[t]
                        m = P.mean(0)
                        v[t] = {"periphery": cancan.periphery(P, per),
                                "center": cancan.center_value(P, cancan.center_index(m, cen)),
                                "trough": cancan.trough_value(P, cancan.side_minima(m), tro)}
                    row = {"layout": lay, "bg": bg, "center": cen, "periphery": per, "trough": tro}
                    for place in ["periphery", "center", "trough"]:
                        a, b, c = (v[t][place] for t in TREATMENTS)
                        row[f"{place}_means"] = f"{a.mean():.0f} / {b.mean():.0f} / {c.mean():.0f}"
                        row[f"{place}_order"] = a.mean() < b.mean() < c.mean()
                        row[f"{place}_p"] = stats.ttest_ind(c, a).pvalue
                        row[f"{place}_ratio40to4"] = c.mean() / a.mean()
                    rows.append(row)
allf6a = pd.DataFrame(rows)
places = ["periphery", "center", "trough"]
allf6a["order_all"] = allf6a[[f"{p}_order" for p in places]].all(axis=1)
allf6a["n_sig"] = (allf6a[[f"{p}_p" for p in places]] < 0.05).sum(axis=1)
allf6a["outcome"] = np.where(allf6a.order_all & (allf6a.n_sig == 3), "reproduced",
                             np.where(allf6a.order_all & (allf6a.n_sig >= 1), "partially", "not"))
f6a = allf6a[allf6a.bg.isin(cancan.BACKGROUNDS)]
print("Primary reading (r2, b1, c1, e1, t1):")
print(f6a[(f6a.layout == "r2") & (f6a.bg == "b1") & (f6a.center == "c1") & (f6a.periphery == "e1") & (f6a.trough == "t1")].T.to_string())
print("\nReadings of the claims file (backgrounds", cancan.BACKGROUNDS, "):")
print("Combinations by outcome:", f6a.outcome.value_counts().to_dict(), "of", len(f6a))
print("Order 4 < 20 < 40 at all three places in", int(f6a.order_all.sum()), "of", len(f6a))
print("Range over combinations of the 40-over-4 ratio:",
      {p: (round(f6a[f"{p}_ratio40to4"].min(), 2), round(f6a[f"{p}_ratio40to4"].max(), 2)) for p in places})
print("Range over combinations of the t-test p (40 against 4):",
      {p: (float(f6a[f"{p}_p"].min().round(5)), float(f6a[f"{p}_p"].max().round(5))) for p in places})
failed = {p: int(((f6a[f"{p}_p"] >= 0.05) & f6a.order_all).sum()) for p in places}
print("Combinations in which the test fails, by place:", failed)
print("Periphery test failures by layout and periphery reading:")
print(f6a[f6a.periphery_p >= 0.05].groupby(["layout", "periphery"]).size().to_string())
print("\nBy layout and background, combinations reproduced / partially / not:")
print(f6a.groupby(["layout", "bg"]).outcome.value_counts().unstack(fill_value=0).to_string())
v6a = allf6a[allf6a.bg.isin(cancan.VARIANT_BACKGROUNDS)]
print("\nVariant b2-fp (footprint median, not a reading of the claims file): combinations by outcome", v6a.outcome.value_counts().to_dict(), "of", len(v6a))

In [ ]:
# Per dish, primary reading: the values the test uses.
data = table("r2", "b1")
rows = []
for t in TREATMENTS:
    pos, P = data[t]
    m = P.mean(0)
    ci, mins = cancan.center_index(m, "c1"), cancan.side_minima(m)
    for d, p in zip(dishes[t], P):
        rows.append({"treatment": t, "dish": d["dish"], "periphery_e1": round(cancan.periphery(p, "e1")[0], 1),
                     "center_c1": round(cancan.center_value(p, ci)[0], 1), "trough_t1": round(cancan.trough_value(p, mins, "t1")[0], 1),
                     "trough_positions_mm": [round(pos[i], 2) for i in mins]})
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# Variant: ROI extent from the footprints (no pixel size), primary reading (r2, b1, c1, e1, t1).
data = table("r2", "b1", "geometric")
v = {}
for t in TREATMENTS:
    pos, P = data[t]
    m = P.mean(0)
    v[t] = {"periphery": cancan.periphery(P, "e1"), "center": cancan.center_value(P, cancan.center_index(m, "c1")),
            "trough": cancan.trough_value(P, cancan.side_minima(m), "t1")}
for place in ["periphery", "center", "trough"]:
    a, b, c = (v[t][place] for t in TREATMENTS)
    print(f"{place}: means {a.mean():.0f} / {b.mean():.0f} / {c.mean():.0f}, order {a.mean() < b.mean() < c.mean()}, "
          f"40 against 4 p = {stats.ttest_ind(c, a).pvalue:.4f}")

**Result: reproduced.** Under the primary reading (both diagonals, raw counts, the two central
ROIs, the mean of the two ends, the mean of the two side minima), the mean intensity rises from 4
to 20 to 40 packets at the periphery (544, 3240, 9712 counts), the center (674, 2631, 5172) and
the least affected area (479, 1148, 1880), and 40 exceeds 4 at each place by a two-sided Student's
t-test on 3 dishes each (p = 0.016, 0.00076 and 0.00079). The order holds in all 162 combinations
of the claims file's readings; all three tests pass in 72 and at least one in the other 90. Only
the periphery test fails (p up to 0.127): in all 54 combinations on diagonal A alone and in the
36 others that take the periphery from the negative end alone. So the full result needs both
ends, on diagonal B or both diagonals; the other readings give a partial result, and the claims
file accepts a claim that holds under any combination. Saturation of up to 5% of an ROI near the
10Px4 electrodes lowers the 10Px4 periphery, so it works against the order. With the ROI extent
taken from the footprints the primary reading gives the same order and p = 0.035, 0.0008 and
0.0008. The footprint variant b2-fp, outside the readings, gives 24 reproduced and 30 partial of
54.

## fig6b-peak-relative-to-periphery

> The central peak was almost twofold brighter than the periphery for 1Px4, and about the same
> as the periphery for 5Px4 and 10Px4.

Criterion (claims file): normalized center 200 +/- 20% for 1Px4 and 100 +/- 10% for 5Px4 and
10Px4; reproduced when all three are within tolerance under one combination of readings;
partially reproduced when 1Px4 is within tolerance and above both other treatments.

In [ ]:
def fig6b_table(extent):
    rows = []
    for lay in cancan.LAYOUTS:
        for bg in cancan.BACKGROUNDS + cancan.VARIANT_BACKGROUNDS:
            data = table(lay, bg, extent)
            for cen in cancan.CENTERS:
                for per in cancan.PERIPHERIES:
                    for norm in cancan.NORMALIZATIONS:
                        row = {"layout": lay, "bg": bg, "center": cen, "periphery": per, "norm": norm}
                        for t in TREATMENTS:
                            pos, P = data[t]
                            Q = cancan.mean_profile(P, per, norm)
                            ci = cancan.center_index(Q, cen)
                            row[t] = cancan.center_value(Q, ci)[0]
                            if norm == "n1":
                                per_dish = cancan.center_value(P, ci) / cancan.periphery(P, per) * 100
                                row[f"{t}_se"] = per_dish.std(ddof=1) / np.sqrt(len(per_dish))
                        rows.append(row)
    return pd.DataFrame(rows)


def judge_6b(f6b):
    f6b["in_1Px4"] = f6b["1Px4"].between(180, 220)
    f6b["in_5Px4"] = f6b["5Px4"].between(90, 110)
    f6b["in_10Px4"] = f6b["10Px4"].between(90, 110)
    f6b["1Px4_highest"] = (f6b["1Px4"] > f6b["5Px4"]) & (f6b["1Px4"] > f6b["10Px4"])
    f6b["outcome"] = np.where(f6b.in_1Px4 & f6b.in_5Px4 & f6b.in_10Px4, "reproduced",
                              np.where(f6b.in_1Px4 & f6b["1Px4_highest"], "partially", "not"))
    return f6b


allf6b = judge_6b(fig6b_table("camera"))
f6b = allf6b[allf6b.bg.isin(cancan.BACKGROUNDS)]
print(f6b.round(1).to_string(index=False))
print("\nReadings of the claims file (backgrounds", cancan.BACKGROUNDS, "): combinations by outcome", f6b.outcome.value_counts().to_dict(), "of", len(f6b))
for t in TREATMENTS:
    print(f"{t}: normalized center {f6b[t].min():.1f} to {f6b[t].max():.1f}% over combinations, median {f6b[t].median():.1f}%")
print("1Px4 above both others in", int(f6b["1Px4_highest"].sum()), "of", len(f6b))
print("Within tolerance: 1Px4 in", int(f6b.in_1Px4.sum()), "; 5Px4 in", int(f6b.in_5Px4.sum()), "; 10Px4 in", int(f6b.in_10Px4.sum()))
print("\nCombinations that are reproduced or partially reproduced:")
print(f6b[f6b.outcome != "not"].round(1).to_string(index=False))
print("\nBy background, combinations by outcome:", allf6b.groupby("bg").outcome.value_counts().unstack(fill_value=0).to_dict("index"))
print("By layout and center, partial combinations:", f6b[f6b.outcome == "partially"].groupby(["layout", "center"]).size().to_dict())
g6b = judge_6b(fig6b_table("geometric"))
gv = g6b[g6b.bg.isin(cancan.BACKGROUNDS)]
print("\nVariant, ROI extent from the footprints (no pixel size): combinations by outcome", gv.outcome.value_counts().to_dict(), "of", len(gv),
      "; by background", g6b.groupby("bg").outcome.value_counts().unstack(fill_value=0).to_dict("index"))
for t in TREATMENTS:
    print(f"  {t}: {gv[t].min():.1f} to {gv[t].max():.1f}%, median {gv[t].median():.1f}%; within tolerance in", int(gv[f"in_{t}"].sum()))
print("  primary reading (r2, b1, c1, e1, n1):", gv[(gv.layout == "r2") & (gv.bg == "b1") & (gv.center == "c1") & (gv.periphery == "e1") & (gv.norm == "n1")][TREATMENTS].round(1).to_dict("records"))

In [ ]:
# Per dish, primary reading (r2, c1, e1) under each background.
rows = []
for bg in cancan.BACKGROUNDS + cancan.VARIANT_BACKGROUNDS:
    data = table("r2", bg)
    for t in TREATMENTS:
        pos, P = data[t]
        ci = cancan.center_index(P.mean(0), "c1")
        for d, p in zip(dishes[t], P):
            rows.append({"bg": bg, "treatment": t, "dish": d["dish"],
                         "center_pct_of_periphery": round(float(cancan.center_value(p, ci)[0] / cancan.periphery(p, "e1")[0] * 100), 1),
                         "center_pct_neg_end": round(float(cancan.center_value(p, ci)[0] / p[0] * 100), 1),
                         "center_pct_pos_end": round(float(cancan.center_value(p, ci)[0] / p[-1] * 100), 1)})
print(pd.DataFrame(rows).to_string(index=False))

# Per dish, every partial combination: the center ROI is the one the combination reads from the mean profile.
print("\nPartial combinations, per dish (counts after the background is subtracted; the percent is per dish, before averaging):")
rows = []
for _, r in f6b[f6b.outcome == "partially"].iterrows():
    data = table(r.layout, r.bg)
    for t in TREATMENTS:
        pos, P = data[t]
        ci = cancan.center_index(cancan.mean_profile(P, r.periphery, r.norm), r.center)
        for d, p in zip(dishes[t], P):
            c, e = cancan.center_value(p, ci)[0], cancan.periphery(p, r.periphery)[0]
            rows.append({"combination": f"{r.layout} {r.bg} {r.center} {r.periphery} {r.norm}", "treatment": t, "dish": d["dish"],
                         "center_pos_mm": round(float(np.mean(pos[ci])), 2), "center": round(c), "periphery": round(e), "pct": round(100 * c / e, 1),
                         "combination_value": round(r[t], 1)})
pdish = pd.DataFrame(rows)
print(pdish.to_string(index=False))
one = pdish[pdish.treatment == "1Px4"]
print("\n1Px4 per-dish percent over the partial combinations:", one.groupby("dish").pct.agg(["min", "median", "max"]).round(1).to_dict("index"))

# How the 1Px4 value depends on the size of the offset: one offset for every dish, from 0 to 300 counts.
print("\n1Px4 normalized center against a common offset subtracted from every dish (r2 c1 e1 n1, and the partial layouts with c2):")
offs = np.arange(0, 301, 25)
sweep = {}
for lay, cen, per, norm in [("r2", "c1", "e1", "n1"), ("r2", "c2", "e1", "n1"), ("r1-dA", "c2", "e1", "n2"), ("r1-dB", "c2", "e1", "n1")]:
    _, P = table(lay, "b1")["1Px4"]
    vals = []
    for o in offs:
        Q = cancan.mean_profile(P - o, per, norm)
        vals.append(round(cancan.center_value(Q, cancan.center_index(Q, cen))[0], 1))
    sweep[f"{lay} {cen} {per} {norm}"] = vals
print(pd.DataFrame(sweep, index=pd.Index(offs, name="offset")).T.to_string())

**Result: partially reproduced.** Under the primary reading (both diagonals, raw counts, the two
central ROIs, the mean of the two ends, each dish normalized) the center is 124.0% of the
periphery for 1Px4, 86.7% for 5Px4 and 58.7% for 10Px4; per dish 107.2 to 151.6%, 73.2 to 106.2%
and 34.5 to 77.4%. No combination of readings puts all three values within tolerance. 1Px4 is
within 180 to 220% and above both others in 13 of the 108 combinations of the claims file's
readings, which meets its rule for a partial reproduction: 5 with the dark offset b2-dark and 8
with b2-min, none with raw counts, and all on a single diagonal with the peak taken as the
highest ROI between the minima (c2). These rows are fragile. Their center is one ROI 36 um wide,
and across them 1Px4 per dish is 170.2 to 380.1% for dish 23, 75.2 to 145.0% for dish 24 and
126.1 to 279.5% for dish 25. The 1Px4 value on diagonal A with c2, e1 and n2 passes 180% once a
common offset of 50 counts is subtracted, less than every dark offset estimated here; on both
diagonals with c1 it stays below 155% for offsets up to 300 counts. 1Px4 is above both others in
96 of 108. 10Px4 is within 90 to 110% in 14 of 108, and is below its periphery under the primary
reading. With the ROI extent from the footprints, 1 of 108 combinations is partial (raw counts)
and none reproduced (primary 116.1, 105.1 and 68.0%). The variant b2-fp, outside the readings,
gives 7 partial of 36. So the ranking holds, "almost twofold" for 1Px4 is reached only on a single
diagonal with a dark offset subtracted, and the 40-packet center is mostly well below its
periphery, not "about the same".

## fig6-trough-half-of-center

> For all packet numbers the central peak was clearly discernible, and the least affected area of
> the monolayer, between the center and the electrodes, showed about 2-fold lower intensity than
> the center.

Criterion (claims file): the least affected area is the minimum of the mean profile between the
center and the most peripheral ROI on each side, (t1) the mean of the two sides or (t2) each side;
reproduced when center over least affected area is within 1.8 to 2.2 for all three treatments;
partially reproduced when within tolerance for one or two and above 1 for all.

In [ ]:
def trough_table(extent):
    rows = []
    for lay in cancan.LAYOUTS:
        for bg in cancan.BACKGROUNDS + cancan.VARIANT_BACKGROUNDS:
            data = table(lay, bg, extent)
            for cen in cancan.CENTERS:
                for norm in cancan.NORMALIZATIONS:
                    for per in (cancan.PERIPHERIES if norm == "n1" else ("e1",)):
                        for tro in cancan.TROUGHS:
                            row = {"layout": lay, "bg": bg, "center": cen, "norm": norm, "periphery": per, "trough": tro}
                            for t in TREATMENTS:
                                pos, P = data[t]
                                Q = cancan.mean_profile(P, per, norm)
                                ci, mins = cancan.center_index(Q, cen), cancan.side_minima(Q)
                                row[t] = cancan.center_value(Q, ci)[0] / cancan.trough_value(Q, mins, tro)[0]
                            rows.append(row)
    ftr = pd.DataFrame(rows)
    inside = ftr[TREATMENTS].apply(lambda s: s.between(1.8, 2.2))
    ftr["n_within"] = inside.sum(axis=1)
    ftr["all_above_1"] = (ftr[TREATMENTS] > 1).all(axis=1)
    ftr["outcome"] = np.where(ftr.n_within == 3, "reproduced",
                              np.where((ftr.n_within >= 1) & ftr.all_above_1, "partially", "not"))
    keep = ftr.bg.isin(cancan.BACKGROUNDS)
    return ftr, inside, keep


allftr, allinside, keep = trough_table("camera")
ftr, inside = allftr[keep], allinside[keep]
print("Rows with n2 do not depend on the periphery reading and are shown once (e1).")
print(ftr.round(2).to_string(index=False))
print("\nReadings of the claims file (backgrounds", cancan.BACKGROUNDS, "): combinations by outcome", ftr.outcome.value_counts().to_dict(), "of", len(ftr))
for t in TREATMENTS:
    print(f"{t}: center / least affected {ftr[t].min():.2f} to {ftr[t].max():.2f}, median {ftr[t].median():.2f}; within 1.8-2.2 in {int(inside[t].sum())} of {len(ftr)}")
print("Above 1 for all three treatments in", int(ftr.all_above_1.sum()), "of", len(ftr))
print("By background, combinations by outcome:", allftr.groupby("bg").outcome.value_counts().unstack(fill_value=0).to_dict("index"))
print("10Px4 by background, range:", allftr.groupby("bg")["10Px4"].agg(["min", "max"]).round(2).to_dict("index"))
gtr, ginside, gkeep = trough_table("geometric")
print("\nVariant, ROI extent from the footprints (no pixel size): combinations by outcome", gtr[gkeep].outcome.value_counts().to_dict(), "of", int(gkeep.sum()))
for t in TREATMENTS:
    print(f"  {t}: {gtr[gkeep][t].min():.2f} to {gtr[gkeep][t].max():.2f}, median {gtr[gkeep][t].median():.2f}; within 1.8-2.2 in {int(ginside[gkeep][t].sum())}")

In [ ]:
# Per dish, primary reading (r2, c1, t1) under each background, minima taken from the treatment's mean profile.
rows = []
for bg in cancan.BACKGROUNDS + cancan.VARIANT_BACKGROUNDS:
    data = table("r2", bg)
    for t in TREATMENTS:
        pos, P = data[t]
        m = P.mean(0)
        ci, mins = cancan.center_index(m, "c1"), cancan.side_minima(m)
        for d, p in zip(dishes[t], P):
            rows.append({"bg": bg, "treatment": t, "dish": d["dish"],
                         "center_over_trough_t1": round(float(cancan.center_value(p, ci)[0] / cancan.trough_value(p, mins, "t1")[0]), 2)})
print(pd.DataFrame(rows).pivot_table(index=["treatment", "dish"], columns="bg", values="center_over_trough_t1", sort=False).to_string())

**Result: partially reproduced.** Under the primary reading the center over the least affected
area is 1.40 for 1Px4, 2.16 for 5Px4 and 2.80 for 10Px4. Over the 216 combinations of the claims
file's readings it is 1.32 to 4.04 for 1Px4 (within 1.8 to 2.2 in 52), 1.86 to 4.37 for 5Px4 (in
29) and 2.66 to 10.11 for 10Px4 (in none). No combination puts all three within tolerance; 79 put
one or two within it, with all three above 1 (27 with raw counts, 25 with b2-dark and 27 with
b2-min). The central peak is discernible for every treatment (the ratio is above 1 in all 216
combinations), but the ratio grows with the number of packets, so "about 2-fold for all the
treatments" does not hold at 40 packets; per dish, 10Px4 is 2.52 to 2.92 with raw counts. The ROI
extent from the footprints gives 69 partial and none reproduced (10Px4 2.68 to 9.27); the variant
b2-fp, outside the readings, gives 14 partial of 72 (10Px4 3.05 to 12.06).

## fig6-peak-narrower

> The central peak appeared narrower (sharper) with increasing number of packets.

Criterion (claims file): (w1) full width of the central peak at half its height above the least
affected area (the mean of the two side minima); (w2) the distance between the two minima on
either side of the center. Each on the mean normalized profile, also with LOESS fits of span
0.25 and 0.5. Reproduced when, under a reading, width decreases from 4 to 20 to 40 packets;
partially reproduced when 40 is narrower than 4 but 20 is out of order. The center reading
sets where the half-height walk starts and the peak height (c1: the two central ROIs; c2: the
highest ROI between the minima).

In [ ]:
rows = []
for lay in cancan.LAYOUTS:
    for bg in cancan.BACKGROUNDS + cancan.VARIANT_BACKGROUNDS:
        data = table(lay, bg)
        for norm in cancan.NORMALIZATIONS:
            for per in (cancan.PERIPHERIES if norm == "n1" else ("e1",)):
                for sm in cancan.SMOOTHING:
                    for cen in cancan.CENTERS:
                        row = {"layout": lay, "bg": bg, "norm": norm, "periphery": per, "smoothing": sm, "center": cen}
                        for t in TREATMENTS:
                            pos, P = data[t]
                            Q = cancan.mean_profile(P, per, norm)
                            if sm != "none":
                                Q = cancan.loess(pos, Q, float(sm.split("-")[1]))
                            ci, mins = cancan.center_index(Q, cen), cancan.side_minima(Q)
                            start = ci[0] if len(ci) == 1 else (ci[0] if Q[ci[0]] >= Q[ci[1]] else ci[1])
                            peak = cancan.center_value(Q, ci)[0]
                            base = cancan.trough_value(Q, mins, "t1")[0]
                            row[f"w1_{t}"] = cancan.half_max_width(pos, Q, start, peak, base)
                            row[f"w2_{t}"] = pos[mins[1]] - pos[mins[0]]
                        rows.append(row)
allfw = pd.DataFrame(rows)
for w in ["w1", "w2"]:
    a, b, c = (allfw[f"{w}_{t}"] for t in TREATMENTS)
    allfw[f"{w}_outcome"] = np.where((a > b) & (b > c), "reproduced", np.where(c < a, "partially", "not"))
fw = allfw[allfw.bg.isin(cancan.BACKGROUNDS)]
# w2 does not depend on the center reading: count it once per combination of the other readings.
fw2 = fw[fw.center == "c1"]
print("Widths in mm. Rows with n2 do not depend on the periphery reading and are shown once (e1);"
      " w2 does not depend on the center reading and is counted once.")
print(fw.round(3).to_string(index=False))
for w, f in [("w1", fw), ("w2", fw2)]:
    print(f"\n{w}: combinations by outcome", f[f"{w}_outcome"].value_counts().to_dict(), "of", len(f),
          "; with n1 only:", f[f.norm == "n1"][f"{w}_outcome"].value_counts().to_dict(),
          "; with n2 only:", f[f.norm == "n2"][f"{w}_outcome"].value_counts().to_dict())
    for t in TREATMENTS:
        s = f[f"{w}_{t}"]
        print(f"  {t}: {s.min():.3f} to {s.max():.3f} mm, median {s.median():.3f}; not measurable (NaN) in {int(s.isna().sum())}")
    print("  by smoothing:", f.groupby("smoothing")[f"{w}_outcome"].value_counts().unstack(fill_value=0).to_dict("index"))
    print("  by background:", f.groupby("bg")[f"{w}_outcome"].value_counts().unstack(fill_value=0).to_dict("index"))
print("\nw1 by center reading:", fw.groupby("center")["w1_outcome"].value_counts().unstack(fill_value=0).to_dict("index"))
v = allfw[allfw.bg.isin(cancan.VARIANT_BACKGROUNDS)]
print("Variant b2-fp (not a reading of the claims file): w1", v.w1_outcome.value_counts().to_dict(), "of", len(v),
      "; w2", v[v.center == "c1"].w2_outcome.value_counts().to_dict(), "of", int((v.center == "c1").sum()))

In [ ]:
# The primary reading (r2, b1, n1, e1), per treatment, under both center readings, with the positions of the minima.
data = table("r2", "b1")
rows = []
for sm in cancan.SMOOTHING:
    for cen in cancan.CENTERS:
        for t in TREATMENTS:
            pos, P = data[t]
            Q = cancan.mean_profile(P, "e1", "n1")
            if sm != "none":
                Q = cancan.loess(pos, Q, float(sm.split("-")[1]))
            ci, mins = cancan.center_index(Q, cen), cancan.side_minima(Q)
            start = ci[0] if len(ci) == 1 else (ci[0] if Q[ci[0]] >= Q[ci[1]] else ci[1])
            rows.append({"smoothing": sm, "center": cen, "treatment": t, "peak_pct": round(cancan.center_value(Q, ci)[0], 1),
                         "peak_mm": round(float(np.mean(pos[ci])), 2),
                         "minima_mm": [round(pos[i], 2) for i in mins], "base_pct": round(cancan.trough_value(Q, mins, "t1")[0], 1),
                         "w1_mm": round(cancan.half_max_width(pos, Q, start, cancan.center_value(Q, ci)[0], cancan.trough_value(Q, mins, "t1")[0]), 3),
                         "w2_mm": round(pos[mins[1]] - pos[mins[0]], 3)})
print(pd.DataFrame(rows).to_string(index=False))
# Per dish, w2 on each dish's own LOESS(0.5) normalized profile, to show the spread between dishes.
rows = []
for t in TREATMENTS:
    pos, P = data[t]
    for d, p in zip(dishes[t], P):
        q = cancan.loess(pos, p / cancan.periphery(p, "e1")[0] * 100, 0.5)
        mins = cancan.side_minima(q)
        rows.append({"treatment": t, "dish": d["dish"], "w2_loess0.5_mm": round(pos[mins[1]] - pos[mins[0]], 3)})
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# Variant: ROI extent from the footprints (no pixel size), r2, b1, n1, e1, c1; widths in mm at the assumed scale.
data = table("r2", "b1", "geometric")
for sm in cancan.SMOOTHING:
    w = {}
    for t in TREATMENTS:
        pos, P = data[t]
        Q = cancan.mean_profile(P, "e1", "n1")
        if sm != "none":
            Q = cancan.loess(pos, Q, float(sm.split("-")[1]))
        ci, mins = cancan.center_index(Q, "c1"), cancan.side_minima(Q)
        start = ci[0] if Q[ci[0]] >= Q[ci[1]] else ci[1]
        w[t] = (cancan.half_max_width(pos, Q, start, cancan.center_value(Q, ci)[0], cancan.trough_value(Q, mins, "t1")[0]),
                pos[mins[1]] - pos[mins[0]])
    print(sm, {t: (round(a, 3), round(b, 3)) for t, (a, b) in w.items()}, "(w1, w2)")

**Result: reproduced.** On the primary reading (r2, b1, n1, e1) the width at half height above
the least affected area (w1) is 1.051, 0.545 and 0.488 mm for 4, 20 and 40 packets; with LOESS of
span 0.25 it is 0.800, 0.654 and 0.560 mm, and with span 0.5 0.992, 0.863 and 0.817 mm. Unsmoothed,
w1 depends on the center reading: under c2 it is 0.591, 0.511 and 0.437 mm, still in order, while
the smoothed widths change by less than 0.02 mm. Over the 216 combinations w1 decreases in 182, 40
is narrower than 4 but 20 out of order in 28, and the 6 others, all unsmoothed with c2, are out
of order. The distance between the minima (w2) does not depend on the center reading and is
counted once per combination: it decreases in 71 of 108, with 28 partial and 9 out of order, all
9 unsmoothed. On the primary reading it is 2.367, 1.650 and 1.435 mm with LOESS 0.5, and 1.363,
1.722 and 1.148 mm unsmoothed. Per dish, w2 on LOESS 0.5 is 1.722 to 2.367 mm for 1Px4, 1.435 to
1.650 for 5Px4 and 1.363 to 1.578 for 10Px4: 1Px4 and 10Px4 do not overlap, 5Px4 and 10Px4 do,
and the criterion asks for no test. With the ROI extent from the footprints w1 keeps the order
under all three smoothings. The widths in mm rest on the assumed pixel size; their order does
not.

## Summary

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

| Claim | Data | Outcome |
|---|---|---|
| fig6a-more-packets-more-uptake | dandiset | reproduced |
| fig6b-peak-relative-to-periphery | dandiset | partially reproduced |
| fig6-trough-half-of-center | dandiset | partially reproduced |
| fig6-peak-narrower | dandiset | reproduced |
| fig4-bipolar-cancellation, fig4-twenty-fold-cancellation | dandiset | blocked: data missing |
| fig5a-packet-without-rotation, fig5bc-rotation-focuses-center | dandiset | blocked: data missing |
| fig7ab-amplitude-scales-uptake, fig7-targeting-independent-of-amplitude, fig7-targeting-independent-of-array-size | dandiset | blocked: data missing |
| fig8-packet2-best-targeting, fig8b-normalized-center, fig8-cancellation-predicts-targeting | dandiset | blocked: data missing |
| fig9-gap-fades-periphery, fig9b-six-fold-targeting | dandiset | blocked: data missing |
| fig3-field-simulation | with inputs | blocked: data missing |

**How the analysis differs from the paper's.** The paper measured the images in MetaMorph with
ROIs that it does not deposit; here the ROIs are placed by code on the detected electrode
footprints. The layout, background, center, periphery and normalization order are not fixed by
the paper, and every reading the claims file names is run. The files hold no dark frame, so the
background reading b2 subtracts dark offsets estimated from the darkest blocks of each image or
the darkest pixel near the array; the electrode footprints hold a faint monolayer and are not
used as a background. The pixel size is assumed (0.65 um);
a variant that places the ROIs from the footprints alone gives the same outcomes. The dandiset
has 3 dishes per treatment, against 3 to 4 in the Fig. 6 legend.

**What was found about the dandiset.**

- It holds only the Fig. 6 experiment (9 dishes at 3.2 kV). The single-cell data of Fig. 4 and
  the monolayer data of Figs. 5, 7, 8 and 9 are not in it.
- The files give no pixel size, no electrode positions and no ROIs, and do not say which array
  was used; the array is the small one by its geometry at the camera scale. The dark electrode footprints
  are 1.66 to 2.09 mm across at that scale, wider than the 1.5 mm needles.
- The species is given as NCBITaxon 9913 (*Bos taurus*) for CHO cells (Chinese hamster), the
  age as "P1D", and the archive's summary counts 18 subjects for 9 files and 9 subject ids.
- Two pairs of dishes share a session start time to the second, one pair across treatments; the
  images differ, so no recording is deposited twice.
- The ImageSeries has unit "n.a." and a rate of 2 Hz for a single frame, and the "200ms" in the
  file names is not explained.

In [ ]:
print(f"Notebook ran in {time.time() - started:.0f} s")